# Import Onda A-mode measurements

Run all cells to load the `A-mode*` folders in `data/Measurements - Onda hydrophone data - 2026`.
Requires pandas and an IPython/Jupyter Python kernel. Works when launched from the project root or `src`.

- `average_df`: saved averaged readings, including the detrended amplitude and A-mode envelope.
- `pulses_df`: individual pulse readings, identified by `measurement` and `pulse_index`.
- `metadata_df`: acquisition metadata from each file's comment header.

Each waveform row represents one time sample. Original CSV column names and units are preserved (seconds and volts). The saved average is detrended, whereas pulse amplitudes are the recorded waveforms; they need not match even for a single pulse. Voltage columns are retained. The calibration cells add pressure columns in Pa using the sensitivity at a user-specified frequency for each recording.

In [34]:
from pathlib import Path

import pandas as pd
from IPython.display import display

# Locate the project when the kernel starts in the repository or src directory.
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "data" / "Measurements - Onda hydrophone data - 2026").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Start the notebook from the project root or src directory.")

measurement_root = project_root / "data" / "Measurements - Onda hydrophone data - 2026"
measurement_dirs = sorted(path for path in measurement_root.glob("A-mode*") if path.is_dir())
if not measurement_dirs:
    raise FileNotFoundError(f"No A-mode folders found in {measurement_root}")

[p.name for p in measurement_dirs]

['A-mode_2MHz_focus_10Vpp_20cycles', 'A-mode_7p37MHz_focus_10Vpp_20cycles']

In [35]:
def read_a_mode_csv(path):
    """Read waveform samples and retain the acquisition header separately."""
    metadata = {}
    with path.open(encoding="utf-8-sig") as stream:
        for line in stream:
            if not line.startswith("#"):
                break
            key, separator, value = line[1:].strip().partition(":")
            if separator:
                metadata[key.strip()] = value.strip()

    waveform = pd.read_csv(path, comment="#", encoding="utf-8-sig")
    required = (
        {"Time_s", "Average_Detrended_Amplitude_V", "A_Mode_Envelope_V"}
        if path.name == "a_scan_average.csv" else {"Time_s", "Amplitude_V"}
    )
    missing = required - set(waveform.columns)
    if missing:
        raise ValueError(f"{path}: missing columns {sorted(missing)}")
    for column in required:
        waveform[column] = pd.to_numeric(waveform[column], errors="raise")
    waveform.insert(0, "measurement", path.parent.name)
    waveform.insert(1, "source_file", str(path.relative_to(measurement_root)))
    metadata.update(measurement=path.parent.name,
                    source_file=str(path.relative_to(measurement_root)))
    return waveform, metadata


average_frames, pulse_frames, metadata_rows = [], [], []
for folder in measurement_dirs:
    average_path = folder / "a_scan_average.csv"
    pulse_paths = sorted(folder.glob("a_scan_pulse_*.csv"),
                         key=lambda path: int(path.stem.rsplit("_", 1)[1]))
    if not average_path.is_file() or not pulse_paths:
        raise FileNotFoundError(f"Expected average and per-pulse CSV files in {folder}")

    frame, metadata = read_a_mode_csv(average_path)
    average_frames.append(frame)
    metadata_rows.append({**metadata, "reading_type": "average"})
    for path in pulse_paths:
        frame, metadata = read_a_mode_csv(path)
        pulse_index = int(path.stem.rsplit("_", 1)[1])
        frame.insert(2, "pulse_index", pulse_index)
        pulse_frames.append(frame)
        metadata_rows.append({**metadata, "pulse_index": pulse_index,
                              "reading_type": "pulse"})

average_df = pd.concat(average_frames, ignore_index=True)
pulses_df = pd.concat(pulse_frames, ignore_index=True)
metadata_df = pd.DataFrame(metadata_rows)
metadata_df["pulse_index"] = pd.to_numeric(metadata_df["pulse_index"]).astype("Int64")

In [36]:
# Sample counts and previews of the imported DataFrames.
display(average_df.groupby("measurement", sort=False).size().rename("average_samples").to_frame())
display(pulses_df.groupby(["measurement", "pulse_index"], sort=False).size().rename("pulse_samples").to_frame())
display(average_df.head())
display(pulses_df.head())
display(metadata_df)

,average_samples
measurement,
A-mode_2MHz_focus_10Vpp_20cycles,10002
A-mode_7p37MHz_focus_10Vpp_20cycles,2002


,,pulse_samples
measurement,pulse_index,
A-mode_2MHz_focus_10Vpp_20cycles,1,10002
A-mode_7p37MHz_focus_10Vpp_20cycles,1,2002


,measurement,source_file,Time_s,Average_Detrended_Amplitude_V,A_Mode_Envelope_V
0,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_averag...,-2.020624e-07,-0.002025,0.000051
1,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_averag...,-1.920624e-07,0.005975,0.000047
2,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_averag...,-1.820624e-07,0.001975,0.000053
3,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_averag...,-1.720624e-07,-0.006024,0.000049
4,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_averag...,-1.620624e-07,0.001976,0.000053


,measurement,source_file,pulse_index,Time_s,Amplitude_V
0,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_pulse_...,1,-2.020624e-07,0.000
1,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_pulse_...,1,-1.920624e-07,0.008
2,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_pulse_...,1,-1.820624e-07,0.004
3,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_pulse_...,1,-1.720624e-07,-0.004
4,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_pulse_...,1,-1.620624e-07,0.004


,run_timestamp,run_id,mode,x_mm,y_mm,z_mm,frequency_hz,amplitude_v,cycles_per_pulse,pulse_count,...,live_preview,signal_source,effective_filter_sampling_rate_hz,effective_highpass_cutoff_hz,measurement,source_file,reading_type,pulse_index,pulse_total,scope_sampling_rate_hz
0,2026-09-25 20:02:38,2,INC,0.5,0.0,0.0,2000000.0,10.0,20.0,1,...,False,hardware,100000000.60774711,500000.0,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_averag...,average,<NA>,NaN,NaN
1,2026-09-25 20:02:38,2,INC,0.5,0.0,0.0,2000000.0,10.0,20.0,1,...,False,hardware,NaN,NaN,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_pulse_...,pulse,1,1,100000000.60774711
2,2026-09-25 20:17:25,1,INC,0.0,0.0,0.0,7370000.0,10.0,20.0,1,...,False,hardware,100000000.60774711,500000.0,A-mode_7p37MHz_focus_10Vpp_20cycles,A-mode_7p37MHz_focus_10Vpp_20cycles/a_scan_ave...,average,<NA>,NaN,NaN
3,2026-09-25 20:17:25,1,INC,0.0,0.0,0.0,7370000.0,10.0,20.0,1,...,False,hardware,NaN,NaN,A-mode_7p37MHz_focus_10Vpp_20cycles,A-mode_7p37MHz_focus_10Vpp_20cycles/a_scan_pul...,pulse,1,1,100000000.60774711


## Hydrophone calibration: voltage to pressure

Edit `measurement_frequencies_mhz` below to specify the measurement frequency in **MHz** for each recording. The calibration file is read from that recording's folder (with or without a `.txt` extension).

Use the table's **Sens(V/Pa)** column directly. For a frequency between table entries, linearly interpolate sensitivity between the closest lower and higher frequencies. Frequencies outside the calibrated range raise an error.

Pressure is `amplitude_V / sensitivity_V_per_Pa`. This applies one sensitivity at the specified frequency to the whole waveform, including the envelope; it is a single-frequency conversion, without broadband frequency-response correction. The calibration file includes the AG-2010 preamplifier, so no extra amplifier gain factor is applied.

In [37]:
# USER SETTINGS: measurement frequencies in MHz (edit these values).
measurement_frequencies_mhz = {
    "A-mode_2MHz_focus_10Vpp_20cycles": 2.0,
    "A-mode_7p37MHz_focus_10Vpp_20cycles": 7.37,
}
calibration_basename = "HGL0400-2875_AG-2010-1384-20_xx_OndaCombineCal_20230621-Reza-Oct-2026"

In [38]:
import io
import numpy as np


def read_calibration(path):
    lines = path.read_text(encoding="utf-8-sig").splitlines()
    header_end = next((i for i, line in enumerate(lines)
                       if line.strip().startswith("HEADER_END")), None)
    if header_end is None:
        raise ValueError(f"Missing HEADER_END marker in {path}")
    table = pd.read_csv(
        io.StringIO("\n".join(lines[header_end + 1:])),
        sep=r"\s+", comment="#", header=None,
        names=["frequency_mhz", "sensitivity_db", "sensitivity_v_per_pa",
               "sensitivity_v2_cm2_per_w"],
    )
    table = table.apply(pd.to_numeric, errors="raise").sort_values("frequency_mhz").reset_index(drop=True)
    if (table.empty or not np.isfinite(table.to_numpy()).all()
            or table["frequency_mhz"].duplicated().any()
            or (table["sensitivity_v_per_pa"] <= 0).any()):
        raise ValueError(f"Invalid calibration table in {path}")
    return table


def sensitivity_at_frequency(table, frequency_mhz):
    frequency_mhz = float(frequency_mhz)
    frequencies = table["frequency_mhz"].to_numpy()
    sensitivities = table["sensitivity_v_per_pa"].to_numpy()
    if not np.isfinite(frequency_mhz) or not frequencies[0] <= frequency_mhz <= frequencies[-1]:
        raise ValueError(f"Frequency must be within {frequencies[0]}–{frequencies[-1]} MHz")
    upper = int(np.searchsorted(frequencies, frequency_mhz))
    if frequencies[upper] == frequency_mhz:
        return float(sensitivities[upper]), float(frequencies[upper]), float(frequencies[upper])
    lower = upper - 1
    fraction = (frequency_mhz - frequencies[lower]) / (frequencies[upper] - frequencies[lower])
    sensitivity = sensitivities[lower] + fraction * (sensitivities[upper] - sensitivities[lower])
    return float(sensitivity), float(frequencies[lower]), float(frequencies[upper])


calibration_tables = {}
calibration_rows = []
for folder in measurement_dirs:
    if folder.name not in measurement_frequencies_mhz:
        raise ValueError(f"Specify a frequency in MHz for {folder.name}")
    candidates = [folder / (calibration_basename + ".txt"), folder / calibration_basename]
    calibration_path = next((p for p in candidates if p.is_file()), None)
    if calibration_path is None:
        raise FileNotFoundError(f"Calibration file missing in {folder}")
    table = read_calibration(calibration_path)
    calibration_tables[folder.name] = table
    frequency = measurement_frequencies_mhz[folder.name]
    sensitivity, lower, upper = sensitivity_at_frequency(table, frequency)
    calibration_rows.append({
        "measurement": folder.name, "frequency_mhz": float(frequency),
        "sensitivity_v_per_pa": sensitivity,
        "lower_frequency_mhz": lower, "upper_frequency_mhz": upper,
        "interpolated": lower != upper,
        "calibration_file": str(calibration_path.relative_to(measurement_root)),
    })

calibration_df = pd.DataFrame(calibration_rows)
sensitivity_by_measurement = calibration_df.set_index("measurement")["sensitivity_v_per_pa"]
for frame, columns in [
    (average_df, {"Average_Detrended_Amplitude_V": "Average_Detrended_Amplitude_Pa",
                  "A_Mode_Envelope_V": "A_Mode_Envelope_Pa"}),
    (pulses_df, {"Amplitude_V": "Amplitude_Pa"}),
]:
    frame["sensitivity_v_per_pa"] = frame["measurement"].map(sensitivity_by_measurement)
    for voltage_column, pressure_column in columns.items():
        frame[pressure_column] = frame[voltage_column] / frame["sensitivity_v_per_pa"]

display(calibration_df)
display(average_df.head(), pulses_df.head())

,measurement,frequency_mhz,sensitivity_v_per_pa,lower_frequency_mhz,upper_frequency_mhz,interpolated,calibration_file
0,A-mode_2MHz_focus_10Vpp_20cycles,2.00,0.000003,2.00,2.0,False,A-mode_2MHz_focus_10Vpp_20cycles/HGL0400-2875_...
1,A-mode_7p37MHz_focus_10Vpp_20cycles,7.37,0.000003,7.35,7.4,True,A-mode_7p37MHz_focus_10Vpp_20cycles/HGL0400-28...


,measurement,source_file,Time_s,Average_Detrended_Amplitude_V,A_Mode_Envelope_V,sensitivity_v_per_pa,Average_Detrended_Amplitude_Pa,A_Mode_Envelope_Pa
0,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_averag...,-2.020624e-07,-0.002025,0.000051,0.000003,-767.147538,19.217160
1,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_averag...,-1.920624e-07,0.005975,0.000047,0.000003,2263.253779,17.669830
2,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_averag...,-1.820624e-07,0.001975,0.000053,0.000003,748.200335,19.935737
3,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_averag...,-1.720624e-07,-0.006024,0.000049,0.000003,-2282.004696,18.416265
4,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_averag...,-1.620624e-07,0.001976,0.000053,0.000003,748.396621,20.015186


,measurement,source_file,pulse_index,Time_s,Amplitude_V,sensitivity_v_per_pa,Amplitude_Pa
0,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_pulse_...,1,-2.020624e-07,0.000,0.000003,0.000000
1,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_pulse_...,1,-1.920624e-07,0.008,0.000003,3030.303174
2,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_pulse_...,1,-1.820624e-07,0.004,0.000003,1515.151587
3,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_pulse_...,1,-1.720624e-07,-0.004,0.000003,-1515.151587
4,A-mode_2MHz_focus_10Vpp_20cycles,A-mode_2MHz_focus_10Vpp_20cycles/a_scan_pulse_...,1,-1.620624e-07,0.004,0.000003,1515.151587


In [39]:
# Example: select one recording and its first pulse for further processing.
# 2MHz data: measurement_dirs[0]
# 7.37 MHz data: measurement_dirs[1]

measurement = measurement_dirs[1].name
average = average_df.loc[average_df["measurement"].eq(measurement)].copy()
pulse = pulses_df.loc[
    pulses_df["measurement"].eq(measurement) & pulses_df["pulse_index"].eq(1)
].copy()
display(average.head(), pulse.head())

,measurement,source_file,Time_s,Average_Detrended_Amplitude_V,A_Mode_Envelope_V,sensitivity_v_per_pa,Average_Detrended_Amplitude_Pa,A_Mode_Envelope_Pa
10002,A-mode_7p37MHz_focus_10Vpp_20cycles,A-mode_7p37MHz_focus_10Vpp_20cycles/a_scan_ave...,0.00004,0.003104,0.000173,0.000003,1155.462606,64.374513
10003,A-mode_7p37MHz_focus_10Vpp_20cycles,A-mode_7p37MHz_focus_10Vpp_20cycles/a_scan_ave...,0.00004,0.003104,0.000219,0.000003,1155.637074,81.503803
10004,A-mode_7p37MHz_focus_10Vpp_20cycles,A-mode_7p37MHz_focus_10Vpp_20cycles/a_scan_ave...,0.00004,0.001105,0.000156,0.000003,411.209869,57.965861
10005,A-mode_7p37MHz_focus_10Vpp_20cycles,A-mode_7p37MHz_focus_10Vpp_20cycles/a_scan_ave...,0.00004,0.005105,0.000243,0.000003,1900.587684,90.497316
10006,A-mode_7p37MHz_focus_10Vpp_20cycles,A-mode_7p37MHz_focus_10Vpp_20cycles/a_scan_ave...,0.00004,-0.012895,0.000138,0.000003,-4800.652910,51.214335


,measurement,source_file,pulse_index,Time_s,Amplitude_V,sensitivity_v_per_pa,Amplitude_Pa
10002,A-mode_7p37MHz_focus_10Vpp_20cycles,A-mode_7p37MHz_focus_10Vpp_20cycles/a_scan_pul...,1,0.00004,0.006,0.000003,2233.805020
10003,A-mode_7p37MHz_focus_10Vpp_20cycles,A-mode_7p37MHz_focus_10Vpp_20cycles/a_scan_pul...,1,0.00004,0.006,0.000003,2233.805020
10004,A-mode_7p37MHz_focus_10Vpp_20cycles,A-mode_7p37MHz_focus_10Vpp_20cycles/a_scan_pul...,1,0.00004,0.004,0.000003,1489.203347
10005,A-mode_7p37MHz_focus_10Vpp_20cycles,A-mode_7p37MHz_focus_10Vpp_20cycles/a_scan_pul...,1,0.00004,0.008,0.000003,2978.406694
10006,A-mode_7p37MHz_focus_10Vpp_20cycles,A-mode_7p37MHz_focus_10Vpp_20cycles/a_scan_pul...,1,0.00004,-0.010,0.000003,-3723.008367


In [40]:
# Overlay the selected pulse and saved average in Pa.
import plotly.graph_objects as go

fig = go.Figure()
fig.add_scatter(x=pulse["Time_s"], y=pulse["Amplitude_Pa"],
                mode="lines", name="Pulse 1", opacity=0.7)
fig.add_scatter(x=average["Time_s"], y=average["Average_Detrended_Amplitude_Pa"],
                mode="lines", name="Average (detrended)")
fig.add_scatter(x=average["Time_s"], y=average["A_Mode_Envelope_Pa"],
                mode="lines", name="Envelope")
fig.update_layout(title=measurement, xaxis_title="Time (s)",
                  yaxis_title="Pressure (Pa)", template="plotly_white",
                  hovermode="x unified")
fig.show()

In [41]:
import numpy as np
import scipy.signal as ss
from scipy.fft import rfft, rfftfreq
import plotly.graph_objects as go
from plotly.subplots import make_subplots

hydrophone_bandwidth = 20 #MHz

time_s = pulse["Time_s"].to_numpy(dtype=float)
pressure_pa = pulse["Amplitude_Pa"].to_numpy(dtype=float)

dt = np.median(np.diff(time_s))
if dt <= 0 or not np.allclose(np.diff(time_s), dt, rtol=1e-5, atol=0):
    raise ValueError("Time samples must be uniformly spaced.")

fs = 1 / dt
print(f"Sampling frequency: {fs / 1e6:.3f} MHz, \n dt = {dt * 1e6:.3f} µs")
pressure_pa = ss.detrend(pressure_pa, type="constant")

# Full-pulse FFT: Hann window with coherent-gain amplitude correction.
window = ss.windows.hann(len(pressure_pa), sym=False)
frequency_hz = rfftfreq(len(pressure_pa), d=dt)
amplitude_pa = np.abs(rfft(pressure_pa * window)) / window.sum()

# Convert to a one-sided peak-amplitude spectrum.
amplitude_pa[1:] *= 2
if len(pressure_pa) % 2 == 0:
    amplitude_pa[-1] /= 2  # Nyquist bin is not doubled.

# Time-resolved spectrum.
nperseg = min(256, len(pressure_pa))
f, t, magnitude = ss.spectrogram(
    pressure_pa,
    fs=fs,
    window="hann",
    nperseg=nperseg,
    noverlap=nperseg // 2,
    nfft=512,
    detrend=False,
    scaling="spectrum",
    mode="magnitude",
)

# Spectrogram level relative to its maximum.
reference = magnitude.max()
level_db = 20 * np.log10(
    np.maximum(magnitude / reference, 1e-6)
) if reference > 0 else np.full_like(magnitude, -120.0)

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=("Full-pulse FFT", "Spectrogram"),
    vertical_spacing=0.18,
)

fig.add_trace(
    go.Scatter(
        x=frequency_hz / 1e6,
        y=amplitude_pa / 1000,
        mode="lines",
        name="FFT",
    ),
    row=1, col=1,
)

fig.add_trace(
    go.Heatmap(
        x=(t + time_s[0]) * 1e6,
        y=f / 1e6,
        z=level_db,
        colorscale="Viridis",
        zmin=-60, zmax=0,
        colorbar=dict(title="dB re. max", len=0.4, y=0.2),
    ),
    row=2, col=1,
)

fig.update_xaxes(title_text="Frequency (MHz)", row=1, col=1)
fig.update_yaxes(title_text="Peak amplitude (kPa)", row=1, col=1)
fig.update_xaxes(title_text="Time (µs)", row=2, col=1)
fig.update_yaxes(title_text="Frequency (MHz)", row=2, col=1)

fig.update_layout(
    title=f"{measurement} — Pulse spectrum",
    template="plotly_white",
    width=1000,
    height=750,
    showlegend=False,
)

# Display only frequencies within the hydrophone's 20 MHz bandwidth.
fig.update_xaxes(range=[0, hydrophone_bandwidth], row=1, col=1)  # FFT frequency axis
fig.update_yaxes(range=[0, hydrophone_bandwidth], row=2, col=1)  # Spectrogram frequency axis

fig.show()

Sampling frequency: 100.000 MHz, 
 dt = 0.010 µs


## A-mode aperture correction and cumulative-energy intensity

A single A-mode trace cannot identify spatial averaging from its 400 µm electrode aperture alone. Supply four measured off-axis/on-axis pressure-like ratios or use the **matching pressure-field scan** below. Scan-derived factors are provisional: the grid is coarse, the effective aperture is assumed, and the A-mode recording must correspond to the same beam geometry and spatial peak. The notebook cannot establish that correspondence automatically. A field-wide or nonlinear waveform correction is not inferred.

Under the user-assumed linear, separable beam, use **one scalar pressure factor** from the spatial profile of `sqrt(PII)`: `K = (3 − beta)/2`, with beta averaged at ± one effective radius in two perpendicular directions. Apply this factor to the selected pulse, average and envelope. This produces a model-corrected waveform estimate; using different compressional/rarefactional peak factors piecewise would distort the waveform. Intensity/PII then scale by K² and cumulative-energy PD is unchanged.

The quadratic method is associated with IEC 62127-1:2007 + Amendment 1:2013 Annex E; this is not a 2022-edition compliance claim. [Derivation](https://pmc.ncbi.nlm.nih.gov/articles/PMC6935508/), [Onda electrode specifications](https://www.ondacorp.com/capsule-hydrophone/).

Intensity uses `I(t) = p²/(rho*c)`; full gated PII and central-98% PII are separate; `PD = 1.25*(t90 − t10)`; `I_PA = selected_PII/PD`; repeating-train `I_TA = selected_PII*PRF`. These are **point quantities**, not I_SPPA/I_SPTA unless this location is established as the spatial peak. Finite-exposure averaging is also reported.

The gate and pre-pulse noise interval must be checked at the selected frequency. Default gates below suit the observed records, not arbitrary recordings. Remove baseline mean and optionally subtract expected noise energy. Signed estimates are retained; unresolved PD/I_PA are NaN. Monotonic cumulative-energy projection for noise-adjusted quantiles is an explicit statistical estimator, not an IEC-prescribed procedure. Do not clip each squared-pressure-minus-variance sample to zero.

The saved envelope is processed by acquisition software and need not equal the analytic envelope of the unfiltered waveform. Both are labeled separately. Noise-energy subtraction does not reconstruct a noise-free instantaneous pressure waveform.

In [42]:
# USER SETTINGS: selected measurement is set in the preceding selection cell.
measurement_frequency_mhz = float(measurement_frequencies_mhz[measurement])
no_cycles = 20
no_pulses = 1
density_kg_m3 = 998.0
sound_speed_m_s = 1482.0
# The A-mode header records PRF; override this value if acquisition information differs.
selected_metadata = metadata_df.loc[
    metadata_df["measurement"].eq(measurement) & metadata_df["reading_type"].eq("average")
].iloc[0]
prf_hz = float(selected_metadata["prf_hz"])
averaging_duration_s = None  # None: no_pulses / prf_hz.

pulse_gates_by_measurement = {
    "A-mode_7p37MHz_focus_10Vpp_20cycles": (42e-6, 48e-6),
    "A-mode_2MHz_focus_10Vpp_20cycles": (40e-6, 60e-6),
}
noise_windows_by_measurement = {
    "A-mode_7p37MHz_focus_10Vpp_20cycles": (40.05e-6, 42e-6),
    "A-mode_2MHz_focus_10Vpp_20cycles": (35e-6, 39e-6),
}
pulse_gate_s = pulse_gates_by_measurement[measurement]
noise_window_s = noise_windows_by_measurement[measurement]
subtract_noise_energy = True
minimum_signal_to_noise_energy_ratio = 3.0
pii_convention = "full_pulse"  # Or "central_98_percent".

electrode_aperture_diameter_um = 400.0
effective_aperture_diameter_um = None  # Override with effective diameter at this frequency.
# None: derive an estimate from the matching scan. Otherwise supply four ratios in (0,1].
measured_aperture_ratios = None  # [axis1_minus, axis1_plus, axis2_minus, axis2_plus]
pressure_field_folders = {
    "A-mode_7p37MHz_focus_10Vpp_20cycles": "pressure map_7p37MHz_10Vpp_focal plane",
    "A-mode_2MHz_focus_10Vpp_20cycles": "pressure map_2MHz_10Vpp_ focal plane",
}
# This assumption must be physically checked before using the scan-derived factor.
assume_same_beam_and_spatial_peak = True
allow_interpolated_aperture_estimate = True


In [43]:
import scipy.signal as ss
from scipy.optimize import isotonic_regression

def pulse_energy_metrics(pressure, times, impedance, gate=None,
                         noise_variance_pa2=0.0, subtract_noise=False, minimum_energy_snr=3.0):
    """Pulse energy with expected noise subtraction and optional quantile reliability mask.

    Signed cumulative energy is projected monotonically for quantiles only.
    No sample-by-sample clipping of p²-noise variance is performed.
    """
    from scipy.optimize import isotonic_regression
    pressure, times = np.asarray(pressure, float), np.asarray(times, float)
    if pressure.shape != times.shape or len(times) < 2 or not np.isfinite(pressure).all():
        raise ValueError("Expected a finite 1-D waveform and matching times")
    if not np.isfinite(times).all() or (np.diff(times) <= 0).any() or not np.isfinite(impedance) or impedance <= 0:
        raise ValueError("Invalid time axis or acoustic impedance")
    if not np.isfinite(noise_variance_pa2) or noise_variance_pa2 < 0:
        raise ValueError("Noise variance must be finite and nonnegative")
    start, end = (times[0], times[-1]) if gate is None else tuple(gate)
    if not np.isfinite([start, end]).all() or not times[0] <= start < end <= times[-1]:
        raise ValueError("Pulse gate must lie inside the acquired time interval")
    intensity = pressure ** 2 / impedance
    gated_time = np.r_[start, times[(times > start) & (times < end)], end]
    gated_intensity = np.interp(gated_time, times, intensity)
    raw_areas = 0.5 * (gated_intensity[:-1] + gated_intensity[1:]) * np.diff(gated_time)
    raw_cumulative = np.r_[0., np.cumsum(raw_areas)]
    noise_intensity = noise_variance_pa2 / impedance
    noise_energy = noise_intensity * (end - start)
    signed_net = float(raw_cumulative[-1] - noise_energy)
    energy_snr = signed_net / noise_energy if noise_energy > 0 else np.inf if raw_cumulative[-1] > 0 else 0.
    signed_cumulative = raw_cumulative - (noise_intensity * (gated_time - start) if subtract_noise else 0.)
    signed_total = float(signed_cumulative[-1])
    # Nonnegative per-point display estimate. Retain signed values for unbiased spatial sums.
    total = max(signed_total, 0.)
    reliable = bool(total > 0 and energy_snr >= minimum_energy_snr)
    result = dict(pii_full=total, pii_98=np.nan,
                  t01=np.nan, t10=np.nan, t90=np.nan, t99=np.nan, pd=np.nan,
                  gate_start=start, gate_end=end,
                  pii_raw=float(raw_cumulative[-1]), pii_signed=signed_total,
                  expected_noise_pii=noise_energy, energy_snr=energy_snr,
                  duration_reliable=float(reliable), cumulative_projection_change=np.nan)
    if not reliable:
        return result
    # Energy after subtraction is signed and need not increase monotonically.
    # Fit a nondecreasing cumulative curve, constrain it to [0,total], and pin its endpoints.
    cumulative = np.clip(isotonic_regression(signed_cumulative).x, 0., total)
    cumulative[0], cumulative[-1] = 0., total
    result["cumulative_projection_change"] = float(np.max(np.abs(cumulative - signed_cumulative)) / total)
    # Invert the projected cumulative curve; fractional sample timing is interpolated.
    targets = np.array([.01, .1, .9, .99]) * total
    segments = np.clip(np.searchsorted(cumulative, targets, side="right") - 1, 0, len(cumulative) - 2)
    fractions = (targets - cumulative[segments]) / (cumulative[segments + 1] - cumulative[segments])
    t01, t10, t90, t99 = gated_time[segments] + fractions * (gated_time[segments + 1] - gated_time[segments])
    result.update(pii_98=.98 * total, t01=t01, t10=t10, t90=t90, t99=t99,
                  pd=1.25 * (t90 - t10))
    return result

def annex_e_peak_estimate(signal_map, axis1, axis2, radius_mm,
                          measured_ratios=None, allow_interpolation=True):
    """Near-axis quadratic correction for a nonnegative pressure-like quantity.

    Returns a spatial-peak estimate only; no correction is applied off-axis.
    """
    values = np.asarray(signal_map, dtype=float)
    axis1, axis2 = np.asarray(axis1, dtype=float), np.asarray(axis2, dtype=float)
    if values.shape != (len(axis1), len(axis2)) or not np.isfinite(values).all() or (values < 0).any():
        raise ValueError("Expected a finite nonnegative map matching the scan axes")
    if not np.isfinite(radius_mm) or radius_mm <= 0:
        raise ValueError("Radius must be finite and positive")
    if len(axis1) < 2 or len(axis2) < 2 or (np.diff(axis1) <= 0).any() or (np.diff(axis2) <= 0).any():
        raise ValueError("Two increasing spatial axes are required")
    i, j = np.unravel_index(np.argmax(values), values.shape)
    peak = float(values[i, j])
    coarse = bool(max(np.diff(axis1).max(), np.diff(axis2).max()) > radius_mm)
    result = {"axis1_index": int(i), "axis2_index": int(j),
              "axis1_mm": float(axis1[i]), "axis2_mm": float(axis2[j]),
              "measured_peak": peak, "radius_mm": radius_mm,
              "scan_coarser_than_radius": coarse,
              "effective_diameter_assumed": effective_aperture_diameter_um is None}
    if peak == 0:
        return {**result, "beta": np.nan, "pressure_factor": 1.0,
                "corrected_peak": 0.0, "status": "zero signal", "source": "none"}
    if measured_ratios is None and np.all(values == peak):
        return {**result, "beta": 1.0, "pressure_factor": 1.0,
                "corrected_peak": peak, "status": "uniform sampled field; no estimated correction",
                "source": "uniform sampled profile"}
    if measured_ratios is not None:
        ratios = np.asarray(measured_ratios, dtype=float)
        if ratios.shape != (4,) or not np.isfinite(ratios).all() or (ratios <= 0).any() or (ratios > 1).any():
            raise ValueError("Supply four ratios in (0, 1], measured around the quantity's peak")
        source = "user-supplied measurements"
    else:
        if coarse and not allow_interpolation:
            raise ValueError("Scan spacing exceeds hydrophone radius; acquire a finer scan or supply measured ratios")
        x, y = axis1[i], axis2[j]
        if x - radius_mm < axis1[0] or x + radius_mm > axis1[-1] or y - radius_mm < axis2[0] or y + radius_mm > axis2[-1]:
            return {**result, "beta": np.nan, "pressure_factor": np.nan,
                    "corrected_peak": np.nan, "status": "insufficient spatial coverage", "source": "none"}
        ratios = np.r_[np.interp([x - radius_mm, x + radius_mm], axis1, values[:, j]),
                       np.interp([y - radius_mm, y + radius_mm], axis2, values[i, :])] / peak
        source = "interpolated scan profile"
    beta = float(ratios.mean())
    factor = (3.0 - beta) / 2.0
    drops_db = -20 * np.log10(np.maximum(ratios, np.finfo(float).tiny))
    status = "quadratic-model estimate"
    if measured_ratios is None and coarse:
        status = "provisional: coarse scan interpolation"
    if (ratios <= 0).any() or (ratios > 1).any():
        status = "invalid off-axis ratios"
        factor = np.nan
    return {**result, "beta": beta, "pressure_factor": factor,
            "corrected_peak": peak * factor, "status": status, "source": source,
            "beta_axis1_minus": ratios[0], "beta_axis1_plus": ratios[1],
            "beta_axis2_minus": ratios[2], "beta_axis2_plus": ratios[3],
            "max_off_axis_drop_db": float(drops_db.max()),
            "off_axis_drop_exceeds_1db": bool((drops_db > 1).any())}


def process_point_waveform(frame, pressure_column, pressure_factor=1.0):
    times = frame["Time_s"].to_numpy(dtype=float)
    values = frame[pressure_column].to_numpy(dtype=float)
    if len(times) < 2 or not np.allclose(np.diff(times), np.median(np.diff(times)), rtol=1e-5, atol=0):
        raise ValueError("Expected uniformly spaced samples")
    if not times[0] <= noise_window_s[0] < noise_window_s[1] <= times[-1]:
        raise ValueError("Baseline window must lie within this waveform")
    if max(noise_window_s[0], pulse_gate_s[0]) < min(noise_window_s[1], pulse_gate_s[1]):
        raise ValueError("Pulse and baseline windows must not overlap")
    noise_mask = (times >= noise_window_s[0]) & (times <= noise_window_s[1])
    if noise_mask.sum() < 20:
        raise ValueError("Insufficient baseline samples")
    baseline = float(np.mean(values[noise_mask]))
    variance = float(np.var(values[noise_mask], ddof=1))
    centred = values - baseline
    result = pulse_energy_metrics(centred * pressure_factor, times, acoustic_impedance_pa_s_m,
                                 pulse_gate_s, variance * pressure_factor**2,
                                 subtract_noise_energy, minimum_signal_to_noise_energy_ratio)
    pii = result["pii_full"] if pii_convention == "full_pulse" else result["pii_98"]
    duration = result["pd"]
    result.update(
        selected_pii_j_m2=pii, selected_pii_mj_cm2=pii / 10,
        pulse_average_intensity_w_cm2=pii / duration / 1e4 if np.isfinite(duration) and duration > 0 else np.nan,
        temporal_average_intensity_w_cm2=pii * prf_hz / 1e4,
        finite_exposure_intensity_w_cm2=no_pulses * pii / temporal_averaging_duration_s / 1e4,
        peak_compressional_kpa=max(float(np.max(centred)), 0) * pressure_factor / 1000,
        peak_rarefactional_magnitude_kpa=max(float(-np.min(centred)), 0) * pressure_factor / 1000,
        baseline_pressure_pa=baseline, noise_variance_pa2=variance * pressure_factor**2,
    )
    return result, centred * pressure_factor


In [44]:
if not all(np.isfinite(value) and value > 0 for value in
           [measurement_frequency_mhz, no_cycles, no_pulses, prf_hz, density_kg_m3,
            sound_speed_m_s, minimum_signal_to_noise_energy_ratio]):
    raise ValueError("Frequency, pulse parameters, water properties and SNR threshold must be positive")
if int(no_pulses) != no_pulses:
    raise ValueError("no_pulses must be an integer")
if pii_convention not in {"full_pulse", "central_98_percent"}:
    raise ValueError("Unknown PII convention")
acoustic_impedance_pa_s_m = density_kg_m3 * sound_speed_m_s
nominal_pulse_duration_s = no_cycles / (measurement_frequency_mhz * 1e6)
temporal_averaging_duration_s = (no_pulses / prf_hz if averaging_duration_s is None
                                else float(averaging_duration_s))
if not np.isfinite(temporal_averaging_duration_s) or temporal_averaging_duration_s <= 0:
    raise ValueError("Exposure averaging duration must be positive")
aperture_diameter_um = (electrode_aperture_diameter_um if effective_aperture_diameter_um is None
                        else float(effective_aperture_diameter_um))
if not np.isfinite(aperture_diameter_um) or aperture_diameter_um <= 0:
    raise ValueError("Effective aperture diameter must be positive")
effective_radius_mm = aperture_diameter_um / 2000

if measured_aperture_ratios is not None:
    ratios = np.asarray(measured_aperture_ratios, dtype=float)
    if ratios.shape != (4,) or not np.isfinite(ratios).all() or (ratios <= 0).any() or (ratios > 1).any():
        raise ValueError("Supply four pressure-like ratios in (0,1]")
    beta = float(ratios.mean())
    aperture_pressure_factor = (3 - beta) / 2
    aperture_diagnostics = {"beta": beta, "pressure_factor": aperture_pressure_factor,
                            "source": "user-supplied ratios", "status": "quadratic-model estimate"}
else:
    if not assume_same_beam_and_spatial_peak:
        raise ValueError("Supply measured ratios unless the A-mode trace shares the scan beam and spatial peak")
    scan_root = measurement_root / pressure_field_folders[measurement]
    scan_manifest_df = pd.read_csv(scan_root / "point_manifest.csv")
    if set(scan_manifest_df["col_axis_role"]) != {"axis_1"} or set(scan_manifest_df["row_axis_role"]) != {"axis_2"}:
        raise ValueError("Expected column axis 1 and row axis 2 in matching scan")
    scan_axis1_mm = np.sort(scan_manifest_df["scan_mm"].unique())
    scan_axis2_mm = np.sort(scan_manifest_df["cross_mm"].unique())
    if len(scan_manifest_df) != len(scan_axis1_mm) * len(scan_axis2_mm) or scan_manifest_df.duplicated(["scan_mm", "cross_mm"]).any():
        raise ValueError("Matching scan must be a complete grid without duplicate coordinates")
    calibration_candidates = [scan_root / (calibration_basename + ".txt"), scan_root / calibration_basename]
    scan_calibration_path = next((path for path in calibration_candidates if path.is_file()), None)
    if scan_calibration_path is None:
        raise FileNotFoundError("Matching scan calibration is missing")
    scan_sensitivity_v_per_pa = sensitivity_at_frequency(read_calibration(scan_calibration_path), measurement_frequency_mhz)[0]
    scan_waveforms = []
    for _, point in scan_manifest_df.iterrows():
        frame = pd.read_csv(scan_root / point["measurement_csv"], comment="#")
        frame = frame.rename(columns={"Time (s)": "Time_s", "Amplitude (V)": "Amplitude_V"})
        frame["Amplitude_Pa"] = frame["Amplitude_V"] / scan_sensitivity_v_per_pa
        scan_waveforms.append(frame)
    reference_times = scan_waveforms[0]["Time_s"].to_numpy()
    overlap_start = max(frame["Time_s"].iloc[0] for frame in scan_waveforms)
    overlap_end = min(frame["Time_s"].iloc[-1] for frame in scan_waveforms)
    scan_time_s = reference_times[(reference_times >= overlap_start) & (reference_times <= overlap_end)]
    scan_sqrt_ppsi_map = np.empty((len(scan_axis1_mm), len(scan_axis2_mm)))
    for (_, point), original in zip(scan_manifest_df.iterrows(), scan_waveforms):
        aligned = pd.DataFrame({"Time_s": scan_time_s,
                                "Amplitude_Pa": np.interp(scan_time_s, original["Time_s"], original["Amplitude_Pa"])})
        result, _ = process_point_waveform(aligned, "Amplitude_Pa")
        i = int(np.searchsorted(scan_axis1_mm, point["scan_mm"]))
        j = int(np.searchsorted(scan_axis2_mm, point["cross_mm"]))
        scan_sqrt_ppsi_map[i, j] = np.sqrt(max(result["selected_pii_j_m2"], 0) * acoustic_impedance_pa_s_m)
    if not np.isfinite(scan_sqrt_ppsi_map).all():
        raise ValueError("Selected PII convention is unresolved at some scan points; use full-pulse PII")
    aperture_diagnostics = annex_e_peak_estimate(
        scan_sqrt_ppsi_map, scan_axis1_mm, scan_axis2_mm, effective_radius_mm,
        allow_interpolation=allow_interpolated_aperture_estimate)
    aperture_diagnostics["scan_folder"] = str(scan_root.relative_to(project_root))
    aperture_diagnostics["assumes_same_beam_and_spatial_peak"] = True
    aperture_pressure_factor = aperture_diagnostics["pressure_factor"]
if not np.isfinite(aperture_pressure_factor) or aperture_pressure_factor < 1:
    raise ValueError("A valid spatial-peak correction factor could not be established")
aperture_intensity_factor = aperture_pressure_factor**2
aperture_correction_df = pd.DataFrame([{**aperture_diagnostics,
                                       "effective_radius_mm": effective_radius_mm,
                                       "intensity_factor": aperture_intensity_factor,
                                       "frequency_mhz": measurement_frequency_mhz}])
display(aperture_correction_df)


,axis1_index,axis2_index,axis1_mm,axis2_mm,measured_peak,radius_mm,scan_coarser_than_radius,effective_diameter_assumed,beta,pressure_factor,...,beta_axis1_plus,beta_axis2_minus,beta_axis2_plus,max_off_axis_drop_db,off_axis_drop_exceeds_1db,scan_folder,assumes_same_beam_and_spatial_peak,effective_radius_mm,intensity_factor,frequency_mhz
0,6,6,2.181818,2.181818,239.573131,0.2,True,True,0.609742,1.195129,...,0.627539,0.573193,0.597519,4.833987,True,data/Measurements - Onda hydrophone data - 202...,True,0.2,1.428333,7.37


In [45]:
# Preserve original DataFrames and store model-corrected, baseline-centred estimates separately.
pulse_corrected = pulse.copy()
average_corrected = average.copy()
intensity_rows = []
for label, frame, column in [
    ("Selected pulse", pulse, "Amplitude_Pa"),
    ("Saved average", average, "Average_Detrended_Amplitude_Pa"),
]:
    raw, centred = process_point_waveform(frame, column)
    corrected, corrected_waveform = process_point_waveform(frame, column, aperture_pressure_factor)
    for kind, result in [("Measured (baseline-centred)", raw), ("Aperture-corrected estimate", corrected)]:
        intensity_rows.append({"trace": label, "processing": kind, **result,
                               "prf_hz": prf_hz, "frequency_mhz": measurement_frequency_mhz,
                               "nominal_pulse_duration_s": nominal_pulse_duration_s,
                               "pii_convention": pii_convention})
    if label == "Selected pulse":
        pulse_corrected["Baseline_Centred_Pressure_Pa"] = centred
        pulse_corrected["Corrected_Amplitude_Pa"] = corrected_waveform
        pulse_corrected["Corrected_Analytic_Envelope_Pa"] = np.abs(ss.hilbert(corrected_waveform))
        pulse_corrected["Corrected_Instantaneous_Intensity_W_m2"] = corrected_waveform**2 / acoustic_impedance_pa_s_m
        pulse_energy_result = corrected
    else:
        average_corrected["Baseline_Centred_Pressure_Pa"] = centred
        average_corrected["Corrected_Average_Pressure_Pa"] = corrected_waveform
        average_corrected["Corrected_Saved_Envelope_Pa"] = frame["A_Mode_Envelope_Pa"].to_numpy() * aperture_pressure_factor
        average_corrected["Corrected_Analytic_Envelope_Pa"] = np.abs(ss.hilbert(corrected_waveform))
        average_corrected["Corrected_Instantaneous_Intensity_W_m2"] = corrected_waveform**2 / acoustic_impedance_pa_s_m

intensity_report_df = pd.DataFrame(intensity_rows)
valid_durations = intensity_report_df["pd"].dropna()
if not valid_durations.empty:
    minimum_exposure = (no_pulses - 1) / prf_hz + valid_durations.max()
    if valid_durations.max() * prf_hz > 1 or temporal_averaging_duration_s < minimum_exposure:
        raise ValueError("PRF or exposure averaging duration is inconsistent with measured PD")
intensity_report_df["PD_us"] = intensity_report_df["pd"] * 1e6
report_columns = ["trace", "processing", "peak_compressional_kpa", "peak_rarefactional_magnitude_kpa",
                  "PD_us", "selected_pii_mj_cm2", "pulse_average_intensity_w_cm2",
                  "temporal_average_intensity_w_cm2", "finite_exposure_intensity_w_cm2",
                  "energy_snr", "duration_reliable"]
display(intensity_report_df[report_columns])

# Individual-pulse metrics: averaging p²-based metrics retains pulse-to-pulse variation.
per_pulse_rows = []
for pulse_index, frame in pulses_df.loc[pulses_df["measurement"].eq(measurement)].groupby("pulse_index"):
    result, _ = process_point_waveform(frame, "Amplitude_Pa", aperture_pressure_factor)
    per_pulse_rows.append({"pulse_index": pulse_index, **result})
per_pulse_intensity_df = pd.DataFrame(per_pulse_rows)
display(per_pulse_intensity_df[["pulse_index", "pd", "selected_pii_mj_cm2",
                               "pulse_average_intensity_w_cm2", "temporal_average_intensity_w_cm2"]])


,trace,processing,peak_compressional_kpa,peak_rarefactional_magnitude_kpa,PD_us,selected_pii_mj_cm2,pulse_average_intensity_w_cm2,temporal_average_intensity_w_cm2,finite_exposure_intensity_w_cm2,energy_snr,duration_reliable
0,Selected pulse,Measured (baseline-centred),191.416089,195.032180,2.646333,0.002574,0.972564,0.000257,0.000257,2226.644147,1.0
1,Selected pulse,Aperture-corrected estimate,228.766924,233.088620,2.646333,0.003676,1.389145,0.000368,0.000368,2226.644147,1.0
2,Saved average,Measured (baseline-centred),191.496169,194.948610,2.646334,0.002574,0.972551,0.000257,0.000257,2228.329926,1.0
3,Saved average,Aperture-corrected estimate,228.862631,232.988743,2.646334,0.003676,1.389126,0.000368,0.000368,2228.329926,1.0


,pulse_index,pd,selected_pii_mj_cm2,pulse_average_intensity_w_cm2,temporal_average_intensity_w_cm2
0,1,0.000003,0.003676,1.389145,0.000368


In [46]:
from plotly.subplots import make_subplots

fig_corrected_a_mode = make_subplots(
    rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.09,
    subplot_titles=("Aperture-corrected pressure estimate", "Corrected envelopes",
                    "Selected pulse cumulative energy"))
fig_corrected_a_mode.add_scatter(x=pulse_corrected["Time_s"] * 1e6,
    y=pulse_corrected["Baseline_Centred_Pressure_Pa"] / 1000,
    name="Measured pulse", opacity=.45, row=1, col=1)
fig_corrected_a_mode.add_scatter(x=pulse_corrected["Time_s"] * 1e6,
    y=pulse_corrected["Corrected_Amplitude_Pa"] / 1000,
    name="Corrected pulse estimate", row=1, col=1)
fig_corrected_a_mode.add_scatter(x=average_corrected["Time_s"] * 1e6,
    y=average_corrected["Corrected_Average_Pressure_Pa"] / 1000,
    name="Corrected average estimate", row=1, col=1)
fig_corrected_a_mode.add_scatter(x=average_corrected["Time_s"] * 1e6,
    y=average_corrected["Corrected_Saved_Envelope_Pa"] / 1000,
    name="Corrected saved envelope", row=2, col=1)
fig_corrected_a_mode.add_scatter(x=pulse_corrected["Time_s"] * 1e6,
    y=pulse_corrected["Corrected_Analytic_Envelope_Pa"] / 1000,
    name="Corrected pulse analytic envelope", row=2, col=1)
fig_corrected_a_mode.add_scatter(x=average_corrected["Time_s"] * 1e6,
    y=average_corrected["Corrected_Analytic_Envelope_Pa"] / 1000,
    name="Corrected average analytic envelope", row=2, col=1)

times = pulse_corrected["Time_s"].to_numpy()
start, end = pulse_gate_s
gated_times = np.r_[start, times[(times > start) & (times < end)], end]
gated_intensity = np.interp(gated_times, times, pulse_corrected["Corrected_Instantaneous_Intensity_W_m2"])
raw_cumulative = np.r_[0., np.cumsum(.5 * (gated_intensity[:-1] + gated_intensity[1:]) * np.diff(gated_times))]
noise_intensity = pulse_energy_result["noise_variance_pa2"] / acoustic_impedance_pa_s_m
signed_cumulative = raw_cumulative - (noise_intensity * (gated_times-start) if subtract_noise_energy else 0)
total = max(float(signed_cumulative[-1]), 0.)
projected = np.clip(isotonic_regression(signed_cumulative).x, 0., total)
projected[0], projected[-1] = 0., total
energy_fraction = projected / total if total > 0 else np.full_like(projected, np.nan)
fig_corrected_a_mode.add_scatter(x=gated_times * 1e6, y=energy_fraction,
                               name="Noise-adjusted cumulative energy", row=3, col=1)
for key in ["t01", "t10", "t90", "t99"]:
    value = pulse_energy_result[key]
    if np.isfinite(value):
        fig_corrected_a_mode.add_vline(x=value * 1e6, line_dash="dot", annotation_text=key, row=3, col=1)
for row in [1, 2]:
    fig_corrected_a_mode.add_vrect(x0=start*1e6, x1=end*1e6, fillcolor="lightgray",
                                  opacity=.12, line_width=0, row=row, col=1)
    fig_corrected_a_mode.update_yaxes(title_text="Pressure (kPa)", row=row, col=1)
fig_corrected_a_mode.update_yaxes(title_text="Energy fraction", row=3, col=1)
fig_corrected_a_mode.update_xaxes(title_text="Time (µs)", range=[start*1e6-1, end*1e6+1], row=3, col=1)
fig_corrected_a_mode.update_layout(
    title=f"{measurement} — provisional aperture factor {aperture_pressure_factor:.4f}",
    template="plotly_white", width=1100, height=950, hovermode="x unified")
fig_corrected_a_mode.show()


## Frequency-dependent sensitivity correction

This cell implements spectral sensitivity scaling:

\[
P(f) = 



rac{V(f)}{M(f)}, \qquad p(t) = \operatorname{irfft}\{P(f)\},
\]

where `M(f)` is the calibration-file sensitivity in V/Pa interpolated onto the positive FFT-frequency bins. The supplied file contains only the magnitude `Sens(V/Pa)`, not calibration phase. Consequently this is a **magnitude-only frequency correction**, with the measured voltage phase retained. A full complex deconvolution requires complex sensitivity, including phase, and appropriate regularization.

Only 1–20 MHz is supported by the supplied calibration. The code uses a smooth taper inside a configurable passband to avoid a sharp frequency truncation. It does not extrapolate sensitivity beyond calibration. The optional aperture factor is applied after spectral scaling as a scalar spatial-peak estimate; it does not perform a spatial deconvolution.

In [47]:
# USER SETTINGS: frequency-dependent sensitivity correction.
frequency_correction_passband_mhz = (1.0, 20.0)  # Must lie within the calibration table.
edge_taper_mhz = 0.25  # Raised-cosine taper applied inside both passband edges.
remove_dc_before_fft = True
apply_aperture_factor_after_frequency_correction = True


def raised_cosine_passband(frequency_hz, lower_hz, upper_hz, taper_hz):
    """Return a unity-centred passband tapered to zero at calibrated limits."""
    if not (0 <= taper_hz and lower_hz < upper_hz and 2 * taper_hz < upper_hz - lower_hz):
        raise ValueError("Invalid frequency passband or taper")
    gain = np.zeros_like(frequency_hz, dtype=float)
    flat = (frequency_hz >= lower_hz + taper_hz) & (frequency_hz <= upper_hz - taper_hz)
    gain[flat] = 1.0
    if taper_hz == 0:
        gain[(frequency_hz >= lower_hz) & (frequency_hz <= upper_hz)] = 1.0
        return gain
    low = (frequency_hz >= lower_hz) & (frequency_hz < lower_hz + taper_hz)
    high = (frequency_hz > upper_hz - taper_hz) & (frequency_hz <= upper_hz)
    gain[low] = 0.5 * (1 - np.cos(np.pi * (frequency_hz[low] - lower_hz) / taper_hz))
    gain[high] = 0.5 * (1 + np.cos(np.pi * (frequency_hz[high] - (upper_hz - taper_hz)) / taper_hz))
    return gain


def frequency_correct_pressure(time_s, voltage_v, calibration_table, passband_mhz, taper_mhz,
                               remove_dc=True):
    """Magnitude-calibrate a real voltage trace with rFFT/interpolated V/Pa data."""
    time_s = np.asarray(time_s, dtype=float)
    voltage_v = np.asarray(voltage_v, dtype=float)
    if time_s.ndim != 1 or voltage_v.shape != time_s.shape or len(time_s) < 2:
        raise ValueError("Expected matching one-dimensional time and voltage arrays")
    sample_interval_s = float(np.median(np.diff(time_s)))
    if sample_interval_s <= 0 or not np.allclose(np.diff(time_s), sample_interval_s, rtol=1e-5, atol=0):
        raise ValueError("Time axis must be uniformly increasing")
    if not np.isfinite(voltage_v).all():
        raise ValueError("Voltage waveform contains non-finite values")

    calibration_frequency_hz = calibration_table["frequency_mhz"].to_numpy(float) * 1e6
    calibration_sensitivity_v_per_pa = calibration_table["sensitivity_v_per_pa"].to_numpy(float)
    lower_hz, upper_hz = np.asarray(passband_mhz, dtype=float) * 1e6
    if not (calibration_frequency_hz[0] <= lower_hz < upper_hz <= calibration_frequency_hz[-1]):
        raise ValueError(
            f"Passband must be within the calibration range "
            f"{calibration_frequency_hz[0] / 1e6:g}–{calibration_frequency_hz[-1] / 1e6:g} MHz"
        )

    input_voltage_v = voltage_v - voltage_v.mean() if remove_dc else voltage_v.copy()
    frequency_hz = np.fft.rfftfreq(len(input_voltage_v), d=sample_interval_s)
    voltage_spectrum_v = np.fft.rfft(input_voltage_v)
    sensitivity_v_per_pa = np.interp(frequency_hz, calibration_frequency_hz,
                                     calibration_sensitivity_v_per_pa,
                                     left=np.nan, right=np.nan)
    passband_gain = raised_cosine_passband(frequency_hz, lower_hz, upper_hz, taper_mhz * 1e6)
    in_passband = passband_gain > 0
    pressure_spectrum_pa = np.zeros_like(voltage_spectrum_v)
    pressure_spectrum_pa[in_passband] = (
        voltage_spectrum_v[in_passband] / sensitivity_v_per_pa[in_passband] * passband_gain[in_passband]
    )
    # if area_correction_factor != 1.0:
    #     pressure_spectrum_pa[in_passband] *= area_correction_factor[in_passband]

    pressure_pa = np.fft.irfft(pressure_spectrum_pa, n=len(input_voltage_v))
    return {
        "pressure_pa": pressure_pa,
        "frequency_hz": frequency_hz,
        "voltage_spectrum_v": voltage_spectrum_v,
        "pressure_spectrum_pa": pressure_spectrum_pa,
        "sensitivity_v_per_pa": sensitivity_v_per_pa,
        "passband_gain": passband_gain,
        "sampling_rate_hz": 1 / sample_interval_s,
    }


In [48]:

selected_calibration_table = calibration_tables[measurement]
aperture_factor_for_frequency_correction = (
    aperture_pressure_factor if apply_aperture_factor_after_frequency_correction else 1.0
)
if not np.isfinite(aperture_factor_for_frequency_correction) or aperture_factor_for_frequency_correction <= 0:
    raise ValueError("Aperture factor must be finite and positive")

pulse_frequency_result = frequency_correct_pressure(
    pulse["Time_s"], pulse["Amplitude_V"], selected_calibration_table,
    frequency_correction_passband_mhz, edge_taper_mhz, remove_dc_before_fft,
)
average_frequency_result = frequency_correct_pressure(
    average["Time_s"], average["Average_Detrended_Amplitude_V"], selected_calibration_table,
    frequency_correction_passband_mhz, edge_taper_mhz, remove_dc_before_fft,
)

pulse_frequency_corrected = pulse.copy()
pulse_frequency_corrected["Frequency_Corrected_Pressure_Pa"] = (
    pulse_frequency_result["pressure_pa"] * aperture_factor_for_frequency_correction
)
pulse_frequency_corrected["Frequency_Corrected_Envelope_Pa"] = np.abs(
    ss.hilbert(pulse_frequency_corrected["Frequency_Corrected_Pressure_Pa"])
)
average_frequency_corrected = average.copy()
average_frequency_corrected["Frequency_Corrected_Pressure_Pa"] = (
    average_frequency_result["pressure_pa"] * aperture_factor_for_frequency_correction
)
average_frequency_corrected["Frequency_Corrected_Envelope_Pa"] = np.abs(
    ss.hilbert(average_frequency_corrected["Frequency_Corrected_Pressure_Pa"])
)

frequency_correction_summary_df = pd.DataFrame([{
    "measurement": measurement,
    "calibration_range_mhz": f"{selected_calibration_table.frequency_mhz.min():g}–{selected_calibration_table.frequency_mhz.max():g}",
    "passband_mhz": f"{frequency_correction_passband_mhz[0]:g}–{frequency_correction_passband_mhz[1]:g}",
    "edge_taper_mhz": edge_taper_mhz,
    "sampling_rate_mhz": pulse_frequency_result["sampling_rate_hz"] / 1e6,
    "aperture_factor_applied": aperture_factor_for_frequency_correction,
    "sensitivity_phase_available": False,
    "method": "magnitude-only FFT sensitivity scaling",
}])
display(frequency_correction_summary_df)


,measurement,calibration_range_mhz,passband_mhz,edge_taper_mhz,sampling_rate_mhz,aperture_factor_applied,sensitivity_phase_available,method
0,A-mode_7p37MHz_focus_10Vpp_20cycles,1–20,1–20,0.25,100.000001,1.195129,False,magnitude-only FFT sensitivity scaling


In [49]:
from plotly.subplots import make_subplots

fig_frequency_corrected_a_mode = make_subplots(
    rows=3, cols=1, shared_xaxes=False, vertical_spacing=0.12,
    subplot_titles=(
        "Pressure waveform: scalar vs frequency-dependent calibration",
        "Frequency-corrected envelopes",
        "Calibration sensitivity and applied passband",
    ),
)

fig_frequency_corrected_a_mode.add_scatter(
    x=pulse["Time_s"] * 1e6, y=pulse["Amplitude_Pa"] / 1000,
    name="Pulse: scalar sensitivity", opacity=0.45, row=1, col=1,
)
fig_frequency_corrected_a_mode.add_scatter(
    x=pulse_frequency_corrected["Time_s"] * 1e6,
    y=pulse_frequency_corrected["Frequency_Corrected_Pressure_Pa"] / 1000,
    name="Pulse: frequency corrected", row=1, col=1,
)
fig_frequency_corrected_a_mode.add_scatter(
    x=average["Time_s"] * 1e6,
    y=average["Average_Detrended_Amplitude_Pa"] / 1000,
    name="Average: scalar sensitivity", opacity=0.45, row=1, col=1,
)
fig_frequency_corrected_a_mode.add_scatter(
    x=average_frequency_corrected["Time_s"] * 1e6,
    y=average_frequency_corrected["Frequency_Corrected_Pressure_Pa"] / 1000,
    name="Average: frequency corrected", row=1, col=1,
)

fig_frequency_corrected_a_mode.add_scatter(
    x=pulse_frequency_corrected["Time_s"] * 1e6,
    y=pulse_frequency_corrected["Frequency_Corrected_Envelope_Pa"] / 1000,
    name="Pulse: analytic envelope", row=2, col=1,
)
fig_frequency_corrected_a_mode.add_scatter(
    x=average_frequency_corrected["Time_s"] * 1e6,
    y=average_frequency_corrected["Frequency_Corrected_Envelope_Pa"] / 1000,
    name="Average: analytic envelope", row=2, col=1,
)

frequency_mhz = pulse_frequency_result["frequency_hz"] / 1e6
fig_frequency_corrected_a_mode.add_scatter(
    x=selected_calibration_table["frequency_mhz"],
    y=selected_calibration_table["sensitivity_v_per_pa"],
    name="Sensitivity (V/Pa)", row=3, col=1,
)
fig_frequency_corrected_a_mode.add_scatter(
    x=frequency_mhz,
    y=pulse_frequency_result["passband_gain"] * selected_calibration_table["sensitivity_v_per_pa"].max(),
    name="Passband gain (scaled)", line=dict(dash="dash"), row=3, col=1,
)

fig_frequency_corrected_a_mode.update_yaxes(title_text="Pressure (kPa)", row=1, col=1)
fig_frequency_corrected_a_mode.update_yaxes(title_text="Envelope (kPa)", row=2, col=1)
fig_frequency_corrected_a_mode.update_yaxes(title_text="Sensitivity (V/Pa)", type="log", row=3, col=1)
fig_frequency_corrected_a_mode.update_xaxes(title_text="Time (µs)", row=1, col=1)
fig_frequency_corrected_a_mode.update_xaxes(title_text="Time (µs)", row=2, col=1)
fig_frequency_corrected_a_mode.update_xaxes(title_text="Frequency (MHz)", range=[0, 20], row=3, col=1)
fig_frequency_corrected_a_mode.update_layout(
    title=(f"{measurement} — frequency-dependent sensitivity correction "
           f"(aperture factor {aperture_factor_for_frequency_correction:.4f})"),
    template="plotly_white", width=1100, height=1000, hovermode="x unified",
)
fig_frequency_corrected_a_mode.show()


## Focal volume dimensions for 1020L calculation - H148 transducer at 1st and 3rd harmonics and Apeture area correction sanity check   

### 1. Check the IEC 62127-1 Beam-to-Element Ratio Criterion:

Before executing a manual correction, calculate the beam width to see if standard IEC correction models apply:
1. Measure the -6 dB beam width $w_{-6\text{dB}}$ 
2. Determine the ratio: $R_{bh}=\frac{w_{-6\text{dB}}}{d_{\text{active}}}$
3. If $R_{bh} \ge 1.5$: we can use the standard simplified IEC correction factor methods - used above.
4. If $R_{bh} < 1.5$: The spatial averaging error is critically high. IEC standards suggest transitioning to a smaller hydrophone aperture (e.g., 100 um) rather than relying solely on post-processing, as phase cancellation across a$2\lambda$ face will destroy high-frequency components

**Note:** Because intensity is proportional to the square of acoustic pressure $p^{2}$ spatial averaging errors compound dramatically. If your pressure is underestimated by 15% due to the 400 um active diameter hydrophone, $I_{SPPA}$ and $I_{SPTA}$ will be underestimated by roughly 28% before correction.


1. The focal volume dimensions are: (use Sonic concept data sheet for exact numbers, ballpark figures can be taken from 
https://sonicconcepts.com/transducer-selection-guide-single-element/)
2. The values listed below are -3 dB drop width values, slightly smaller than -6dB (FWHM) values required for $R_{bh}$ calculations but at tightly focused transducers, the changes are not substantial. FWHM values however can be obtained from simulations or estimated using the eq. given in Wear paper, see below.


**At fundamental (2.0 MHz) frequency:** the focal volume size is: 0.7600 mm in width and 5.7200 mm in length.
Equivalent to 760 um in width and 5720 um in length. 

**At the 3rd harmonic (7.37 MHz) frequency:** the transducer's focal volume is: 0.210 mm in width and 1.580 mm in length,
Equivalent to 210 um in width and 1580 um in length. 


In [50]:
lambda_f0_um = sound_speed_m_s /2
lambda_3f0_um = sound_speed_m_s /7.37
d_active_um_nominal = 400 # nominal active diameter of the hydrophone element in micrometers
print(f"wavelength at f0: {lambda_f0_um:.1f} µm, wavelength at 3f0: {lambda_3f0_um:.1f} µm, nominal hydrophone active diameter: {d_active_um_nominal} µm")

wavelength at f0: 741.0 µm, wavelength at 3f0: 201.1 µm, nominal hydrophone active diameter: 400 µm


In [51]:
beamwidth_f0_3dB = 760 #um
beamwidth_3f0_3dB = 210 #um
R_bh_f0 = beamwidth_f0_3dB / d_active_um_nominal
R_bh_3f0 = beamwidth_3f0_3dB / d_active_um_nominal   
print(f"Beamwidth to hydrophone diameter ratio at f0: {R_bh_f0:.3f}")
print(f"Beamwidth to hydrophone diameter ratio at 3f0: {R_bh_3f0:.3f}")

Beamwidth to hydrophone diameter ratio at f0: 1.900
Beamwidth to hydrophone diameter ratio at 3f0: 0.525


Therefore, the pressure measurements at 7.37 MHz needs to be repeated using a smaller needle hydrophone and the converted values are not reliable as they are biased. See the following papers for details: 
1. Alves et al, 'Empirical correction method for spatial averaging effect in
ultrasonic device calibration: Enhancing the precision-sensitivity trade-of', JASA 2026 https://doi.org/10.1121/10.0042356
2. Keith Wear, 'A 15-Min Method for Hydrophone Spatial Averaging Correction in Transcranial Neuromodulation System Characterization', UMB 2026 https://doi.org/10.1016/j.ultrasmedbio.2025.09.005
3. 

In [52]:
beamwidth_f0_6dB = 1.41*lambda_f0_um*(63.2/64) #um,  Eq. 7 in Keith Wear paper
beamwidth_3f0_6dB = 1.41*lambda_3f0_um*(63.2/64) #um, Eq. 7 in Keith Wear paper

R_bh_f0 = beamwidth_f0_6dB / d_active_um_nominal
R_bh_3f0 = beamwidth_3f0_6dB / d_active_um_nominal   
print(f"Beamwidth to hydrophone diameter ratio at f0: {R_bh_f0:.3f}")
print(f"Beamwidth to hydrophone diameter ratio at 3f0: {R_bh_3f0:.3f}")

Beamwidth to hydrophone diameter ratio at f0: 2.579
Beamwidth to hydrophone diameter ratio at 3f0: 0.700


In [53]:
beamwidth_3f0_6dB

279.98639755766624

In [54]:
A = 0.94 
B = 5.1
radius_nominal = d_active_um_nominal /2

selected_calibration_table["wavenumber_rad_um"] = (
    2 * np.pi
    * selected_calibration_table["frequency_mhz"].to_numpy()
    /sound_speed_m_s
)
selected_calibration_table["wavelength_um"] = sound_speed_m_s / selected_calibration_table["frequency_mhz"].to_numpy()

display(selected_calibration_table[[
    "frequency_mhz",
    "sensitivity_v_per_pa",
    "wavenumber_rad_um",
]])

,frequency_mhz,sensitivity_v_per_pa,wavenumber_rad_um
0,1.00,0.000003,0.004240
1,1.05,0.000003,0.004452
2,1.10,0.000003,0.004664
3,1.15,0.000003,0.004876
4,1.20,0.000003,0.005088
...,...,...,...
376,19.80,0.000003,0.083945
377,19.85,0.000003,0.084157
378,19.90,0.000003,0.084369
379,19.95,0.000003,0.084581


In [55]:
radius_eff_um = radius_nominal + A * np.exp(-1*selected_calibration_table["wavenumber_rad_um"] * radius_nominal/B) / selected_calibration_table["wavenumber_rad_um"]

In [56]:
beamwidth_6dB = 1.41*selected_calibration_table["wavelength_um"] *(63.2/64) #um,  Eq. 7 in Keith Wear paper
std_bw = beamwidth_6dB / (2*np.sqrt(2*np.log(2)))
Omega_sq = radius_eff_um**2 / (2*std_bw**2)
reduction_factor_S = (1-np.exp(-Omega_sq)) / Omega_sq
correction_factor_pressure = 1 / reduction_factor_S
correction_factor_intensity = 1 / reduction_factor_S**2

In [57]:
std_bw_3f0_um = beamwidth_3f0_6dB / (2*np.sqrt(2*np.log(2)))
wavenumber_3f0_um = 2 * np.pi / lambda_3f0_um
radius_eff_3f0_um = radius_nominal + A * np.exp(-1*wavenumber_3f0_um * radius_nominal/B) / wavenumber_3f0_um
Omega_sq_3f0 = radius_eff_3f0_um**2 / (2*std_bw_3f0_um**2)
reduction_factor_S_3f0 = (1-np.exp(-Omega_sq_3f0)) / Omega_sq_3f0
correction_factor_pressure_3f0 = 1 / reduction_factor_S_3f0
correction_factor_intensity_3f0 = 1 / reduction_factor_S_3f0**2

In [58]:
correction_factor_pressure_3f0

np.float64(1.9620562372369912)

In [59]:

aperture_factor_for_frequency_correction_3f0 = correction_factor_pressure_3f0

pulse_frequency_corrected_3f0 = (
    pulse_frequency_result["pressure_pa"] * aperture_factor_for_frequency_correction_3f0
)



In [60]:
from plotly.subplots import make_subplots

fig_frequency_corrected_a_mode = make_subplots(
    rows=1, cols=1, shared_xaxes=False, vertical_spacing=0.12,
    subplot_titles=(
        "Frequency-aperture-corrected envelopes",
    ),
)

fig_frequency_corrected_a_mode.add_scatter(
    x=pulse["Time_s"] * 1e6, y=pulse["Amplitude_Pa"] / 1000,
    name="Pulse: scalar sensitivity", opacity=0.45, row=1, col=1,
)
fig_frequency_corrected_a_mode.add_scatter(
    x=pulse_frequency_corrected["Time_s"] * 1e6,
    y=pulse_frequency_corrected["Frequency_Corrected_Pressure_Pa"] / 1000,
    name="Pulse: frequency corrected", row=1, col=1,
)
fig_frequency_corrected_a_mode.add_scatter(
    x=pulse_frequency_corrected["Time_s"] * 1e6,
    y=pulse_frequency_corrected_3f0 / 1000,
    name="Pulse: frequency & aperture corrected", row=1, col=1,
)

## Intensity from the 3f₀ aperture-corrected waveform

This calculation uses `pulse_frequency_corrected_3f0` directly. It does **not** apply another aperture factor: pressure was already multiplied by `correction_factor_pressure_3f0`, and pressure-squared quantities therefore use `correction_factor_intensity_3f0 = K²` implicitly.

It reports pulse-intensity integral (PII), pulse duration (PD), pulse-average intensity and temporal-average intensity from the corrected waveform. The labels I_SPPA and I_SPTA are appropriate only if this trace is established as the spatial peak. A single point waveform provides energy fluence, not total acoustic power. The optional power estimates require an independently justified effective beam area `A_eff`, where `P = I_peak × A_eff` for the matching temporal convention.

In [61]:
# USER SETTINGS: intensity/power from pulse_frequency_corrected_3f0.
# This waveform is already frequency- and aperture-corrected: do not multiply by K again.
treat_trace_as_spatial_peak = True
peak_intensity_effective_area_mm2 = None  # Supply independently measured/modelled A_eff to estimate power.

corrected_3f0_time_s = pulse["Time_s"].to_numpy(dtype=float)
corrected_3f0_pressure_pa = np.asarray(pulse_frequency_corrected_3f0, dtype=float)
if corrected_3f0_pressure_pa.shape != corrected_3f0_time_s.shape:
    raise ValueError("pulse_frequency_corrected_3f0 must match the selected pulse time axis")
if not np.isfinite(corrected_3f0_pressure_pa).all():
    raise ValueError("pulse_frequency_corrected_3f0 contains non-finite values")
if not np.isclose(correction_factor_intensity_3f0, correction_factor_pressure_3f0**2):
    raise ValueError("The supplied 3f0 intensity factor must equal the pressure factor squared")

noise_mask_3f0 = ((corrected_3f0_time_s >= noise_window_s[0]) &
                  (corrected_3f0_time_s <= noise_window_s[1]))
if noise_mask_3f0.sum() < 20:
    raise ValueError("The baseline window must contain at least 20 samples")
baseline_pressure_3f0_pa = float(corrected_3f0_pressure_pa[noise_mask_3f0].mean())
noise_variance_3f0_pa2 = float(corrected_3f0_pressure_pa[noise_mask_3f0].var(ddof=1))
corrected_3f0_baseline_centred_pa = corrected_3f0_pressure_pa - baseline_pressure_3f0_pa

intensity_3f0_result = pulse_energy_metrics(
    corrected_3f0_baseline_centred_pa,
    corrected_3f0_time_s,
    acoustic_impedance_pa_s_m,
    gate=pulse_gate_s,
    noise_variance_pa2=noise_variance_3f0_pa2,
    subtract_noise=subtract_noise_energy,
    minimum_energy_snr=minimum_signal_to_noise_energy_ratio,
)
selected_pii_3f0_j_m2 = (
    intensity_3f0_result["pii_full"]
    if pii_convention == "full_pulse"
    else intensity_3f0_result["pii_98"]
)
pd_3f0_s = intensity_3f0_result["pd"]
i_pa_3f0_w_m2 = (
    selected_pii_3f0_j_m2 / pd_3f0_s
    if np.isfinite(pd_3f0_s) and pd_3f0_s > 0 else np.nan
)
i_ta_3f0_w_m2 = selected_pii_3f0_j_m2 * prf_hz

metric_prefix = "I_SP" if treat_trace_as_spatial_peak else "I_P"  # spatial-peak or point measurement
if peak_intensity_effective_area_mm2 is None:
    pulse_average_power_3f0_w = np.nan
    temporal_average_power_3f0_w = np.nan
else:
    area_m2 = float(peak_intensity_effective_area_mm2) * 1e-6
    if not np.isfinite(area_m2) or area_m2 <= 0:
        raise ValueError("peak_intensity_effective_area_mm2 must be positive")
    # Valid only when A_eff = integral(I/I_peak)dA is independently established.
    pulse_average_power_3f0_w = i_pa_3f0_w_m2 * area_m2
    temporal_average_power_3f0_w = i_ta_3f0_w_m2 * area_m2

corrected_3f0_intensity_df = pd.DataFrame([{
    "waveform": "pulse_frequency_corrected_3f0",
    "pressure_aperture_factor_K": correction_factor_pressure_3f0,
    "intensity_aperture_factor_K_squared": correction_factor_intensity_3f0,
    "impedance_pa_s_m": acoustic_impedance_pa_s_m,
    "pii_convention": pii_convention,
    "PII_J_m2": selected_pii_3f0_j_m2,
    "PII_mJ_cm2": selected_pii_3f0_j_m2 / 10,
    "PD_us": pd_3f0_s * 1e6,
    f"{metric_prefix}PA_W_cm2": i_pa_3f0_w_m2 / 1e4,
    f"{metric_prefix}TA_W_cm2": i_ta_3f0_w_m2 / 1e4,
    "pulse_average_power_W": pulse_average_power_3f0_w,
    "temporal_average_power_W": temporal_average_power_3f0_w,
    "A_eff_mm2_for_power": peak_intensity_effective_area_mm2,
    "energy_SNR": intensity_3f0_result["energy_snr"],
    "duration_reliable": bool(intensity_3f0_result["duration_reliable"]),
    "notes": ("I_SP labels assume this trace is the spatial peak; power requires independent A_eff"
              if treat_trace_as_spatial_peak else
              "Point intensities only; this trace has not been declared spatial peak"),
}])
display(corrected_3f0_intensity_df)

# Keep the corrected waveform and its instantaneous intensity available for further plots.
pulse_frequency_corrected_3f0_df = pd.DataFrame({
    "Time_s": corrected_3f0_time_s,
    "Frequency_Aperture_Corrected_Pressure_Pa": corrected_3f0_pressure_pa,
    "Baseline_Centred_Pressure_Pa": corrected_3f0_baseline_centred_pa,
    "Instantaneous_Intensity_W_m2": corrected_3f0_baseline_centred_pa**2 / acoustic_impedance_pa_s_m,
})


,waveform,pressure_aperture_factor_K,intensity_aperture_factor_K_squared,impedance_pa_s_m,pii_convention,PII_J_m2,PII_mJ_cm2,PD_us,I_SPPA_W_cm2,I_SPTA_W_cm2,pulse_average_power_W,temporal_average_power_W,A_eff_mm2_for_power,energy_SNR,duration_reliable,notes
0,pulse_frequency_corrected_3f0,1.962056,3.849665,1479036.0,full_pulse,0.098194,0.009819,2.648285,3.707816,0.000982,NaN,NaN,None,6066.111761,True,I_SP labels assume this trace is the spatial p...


## Comparison: IEC Annex E versus 3f₀ aperture correction

The earlier result uses the scan-derived IEC Annex E quadratic factor `K_IEC = (3 − beta)/2`, applied to a scalar-sensitivity pressure waveform. The newer result uses frequency-dependent sensitivity scaling and the user-calculated `correction_factor_pressure_3f0`.

They are not expected to match exactly: the first is a local profile-based estimate from the matching field scan, while the second comes from a frequency/beam-width model at 3f₀. The table reports both factors and their resulting point metrics without treating either as a validated full-field deconvolution.

In [62]:
# Compare the selected-pulse estimates. Both rows use their own baseline/noise/gate processing.
iec_selected_pulse = intensity_report_df.loc[
    intensity_report_df["trace"].eq("Selected pulse")
    & intensity_report_df["processing"].eq("Aperture-corrected estimate")
]
if len(iec_selected_pulse) != 1:
    raise ValueError("Expected exactly one earlier IEC Annex E selected-pulse result")
iec_selected_pulse = iec_selected_pulse.iloc[0]

new_3f0_pressure_peak_kpa = max(float(np.max(corrected_3f0_baseline_centred_pa)), 0.0) / 1000
new_3f0_rarefactional_kpa = max(float(-np.min(corrected_3f0_baseline_centred_pa)), 0.0) / 1000
new_3f0 = corrected_3f0_intensity_df.iloc[0]

comparison_metrics = [
    ("Pressure correction factor K", aperture_pressure_factor, correction_factor_pressure_3f0, "dimensionless"),
    ("Intensity correction factor K²", aperture_intensity_factor, correction_factor_intensity_3f0, "dimensionless"),
    ("Peak compressional pressure", iec_selected_pulse["peak_compressional_kpa"], new_3f0_pressure_peak_kpa, "kPa"),
    ("Peak rarefactional magnitude", iec_selected_pulse["peak_rarefactional_magnitude_kpa"], new_3f0_rarefactional_kpa, "kPa"),
    ("PII", iec_selected_pulse["selected_pii_mj_cm2"], new_3f0["PII_mJ_cm2"], "mJ/cm²"),
    ("Pulse duration", iec_selected_pulse["pd"] * 1e6, new_3f0["PD_us"], "µs"),
    ("Pulse-average intensity", iec_selected_pulse["pulse_average_intensity_w_cm2"], new_3f0["I_SPPA_W_cm2"], "W/cm²"),
    ("Temporal-average intensity", iec_selected_pulse["temporal_average_intensity_w_cm2"], new_3f0["I_SPTA_W_cm2"], "W/cm²"),
]
aperture_correction_comparison_df = pd.DataFrame(
    comparison_metrics,
    columns=["Metric", "IEC Annex E / scalar sensitivity", "3f0 / frequency sensitivity", "Unit"],
)
reference_values = aperture_correction_comparison_df["IEC Annex E / scalar sensitivity"].to_numpy(float)
new_values = aperture_correction_comparison_df["3f0 / frequency sensitivity"].to_numpy(float)
aperture_correction_comparison_df["3f0-to-IEC ratio"] = np.divide(
    new_values, reference_values,
    out=np.full_like(new_values, np.nan), where=reference_values != 0,
)
aperture_correction_comparison_df["Difference (%)"] = (
    aperture_correction_comparison_df["3f0-to-IEC ratio"] - 1
) * 100

display(aperture_correction_comparison_df.style.format({
    "IEC Annex E / scalar sensitivity": "{:.6g}",
    "3f0 / frequency sensitivity": "{:.6g}",
    "3f0-to-IEC ratio": "{:.4f}",
    "Difference (%)": "{:+.2f}",
}))

print(
    f"IEC factor K = {aperture_pressure_factor:.6g}; "
    f"3f0 factor K = {correction_factor_pressure_3f0:.6g}; "
    f"ratio = {correction_factor_pressure_3f0 / aperture_pressure_factor:.6g}."
)


,Metric,IEC Annex E / scalar sensitivity,3f0 / frequency sensitivity,Unit,3f0-to-IEC ratio,Difference (%)
0,Pressure correction factor K,1.19513,1.96206,dimensionless,1.6417,+64.17
1,Intensity correction factor K²,1.42833,3.84966,dimensionless,2.6952,+169.52
2,Peak compressional pressure,228.767,373.686,kPa,1.6335,+63.35
3,Peak rarefactional magnitude,233.089,379.621,kPa,1.6287,+62.87
4,PII,0.00367614,0.00981935,mJ/cm²,2.6711,+167.11
5,Pulse duration,2.64633,2.64829,µs,1.0007,+0.07
6,Pulse-average intensity,1.38915,3.70782,W/cm²,2.6691,+166.91
7,Temporal-average intensity,0.000367614,0.000981935,W/cm²,2.6711,+167.11


IEC factor K = 1.19513; 3f0 factor K = 1.96206; ratio = 1.64171.


In [63]:
reference_voltage_vpp = 10.0
target_voltage_vpp = 1.28

pressure_scale = target_voltage_vpp / reference_voltage_vpp
intensity_scale = pressure_scale**2

pulse_frequency_corrected_3f0_at_target_pa = (
    pulse_frequency_corrected_3f0 * pressure_scale
)

target_3f0_estimate_df = corrected_3f0_intensity_df.copy()
for column in [
    "PII_J_m2",
    "PII_mJ_cm2",
    "I_SPPA_W_cm2",
    "I_SPTA_W_cm2",
    "pulse_average_power_W",
    "temporal_average_power_W",
]:
    if column in target_3f0_estimate_df:
        target_3f0_estimate_df[column] *= intensity_scale

target_3f0_estimate_df["pressure_aperture_factor_K"] = correction_factor_pressure_3f0
target_3f0_estimate_df["intensity_aperture_factor_K_squared"] = correction_factor_intensity_3f0
target_3f0_estimate_df["target_voltage_vpp"] = target_voltage_vpp

display(target_3f0_estimate_df)

,waveform,pressure_aperture_factor_K,intensity_aperture_factor_K_squared,impedance_pa_s_m,pii_convention,PII_J_m2,PII_mJ_cm2,PD_us,I_SPPA_W_cm2,I_SPTA_W_cm2,pulse_average_power_W,temporal_average_power_W,A_eff_mm2_for_power,energy_SNR,duration_reliable,notes,target_voltage_vpp
0,pulse_frequency_corrected_3f0,1.962056,3.849665,1479036.0,full_pulse,0.001609,0.000161,2.648285,0.060749,0.000016,NaN,NaN,None,6066.111761,True,I_SP labels assume this trace is the spatial p...,1.28


## Acoustic power estimated from focal beamwidth

Estimate acoustic power by assuming a separable Gaussian intensity profile at the focal plane. For full widths measured at −6 dB in **pressure amplitude** (equivalently 25% of peak intensity),

$$
\begin{aligned}
 A_\mathrm{eff}=\iint \dfrac{I(x,y)}{I_\mathrm{peak}}\,dx\,dy = \dfrac{\pi d_xd_y}{8\ln 2}, \\
P_\mathrm{PA}=I_\mathrm{SPPA}A_\mathrm{eff},\\
P_\mathrm{TA}=I_\mathrm{SPTA}A_\mathrm{eff}.
\end{aligned}
$$

`d_x` and `d_y` must be full pressure/PII −6 dB widths at the same focus and drive condition. If the entered widths are intensity FWHM values instead, select that definition; then the area is $\pi d_x d_y / (4\ln2)$. This is a model-based power estimate, not a spatial PII integration. It assumes the Gaussian profile extends over the whole transverse field and that this trace is the spatial peak.

In [64]:
# USER SETTINGS: focal beamwidths for the 3f0 corrected measurement.
beamwidth_x_um = float(beamwidth_3f0_6dB)  # Full width in the first transverse direction.
beamwidth_y_um = float(beamwidth_3f0_6dB)  # Set separately for an elliptical focus.
beamwidth_definition = "pressure_or_pii_minus_6dB"  # Or "intensity_fwhm".

if not all(np.isfinite(value) and value > 0 for value in [beamwidth_x_um, beamwidth_y_um]):
    raise ValueError("Both focal beamwidths must be finite and positive")

if beamwidth_definition == "pressure_or_pii_minus_6dB":
    # This matches a full width where p/p_peak = 0.5 and I/I_peak = 0.25.
    effective_area_denominator = 8 * np.log(2)
elif beamwidth_definition == "intensity_fwhm":
    # This is a full width where I/I_peak = 0.5.
    effective_area_denominator = 4 * np.log(2)
else:
    raise ValueError("Unknown beamwidth_definition")

effective_focal_area_m2 = (
    np.pi * (beamwidth_x_um * 1e-6) * (beamwidth_y_um * 1e-6)
    / effective_area_denominator
)
effective_focal_area_mm2 = effective_focal_area_m2 * 1e6

reference_isppa_w_m2 = float(corrected_3f0_intensity_df.iloc[0]["I_SPPA_W_cm2"]) * 1e4
reference_ispta_w_m2 = float(corrected_3f0_intensity_df.iloc[0]["I_SPTA_W_cm2"]) * 1e4
voltage_cases = {
    f"Reference ({reference_voltage_vpp:g} Vpp)": (reference_isppa_w_m2, reference_ispta_w_m2),
    f"Target estimate ({target_voltage_vpp:g} Vpp)": (
        reference_isppa_w_m2 * intensity_scale,
        reference_ispta_w_m2 * intensity_scale,
    ),
}
power_estimate_df = pd.DataFrame([
    {
        "voltage_case": label,
        "effective_focal_area_mm2": effective_focal_area_mm2,
        "I_SPPA_W_cm2": isppa_w_m2 / 1e4,
        "I_SPTA_W_cm2": ispta_w_m2 / 1e4,
        "pulse_average_power_W": isppa_w_m2 * effective_focal_area_m2,
        "temporal_average_power_W": ispta_w_m2 * effective_focal_area_m2,
        "beamwidth_x_um": beamwidth_x_um,
        "beamwidth_y_um": beamwidth_y_um,
        "beamwidth_definition": beamwidth_definition,
        "method": "Gaussian focal-plane effective-area estimate",
    }
    for label, (isppa_w_m2, ispta_w_m2) in voltage_cases.items()
])
display(power_estimate_df)

print(
    f"A_eff = {effective_focal_area_mm2:.6g} mm²; "
    "power is estimated from I_peak × A_eff under the stated Gaussian-profile assumption."
)


,voltage_case,effective_focal_area_mm2,I_SPPA_W_cm2,I_SPTA_W_cm2,pulse_average_power_W,temporal_average_power_W,beamwidth_x_um,beamwidth_y_um,beamwidth_definition,method
0,Reference (10 Vpp),0.044413,3.707816,0.000982,0.001647,4.361051e-07,279.986398,279.986398,pressure_or_pii_minus_6dB,Gaussian focal-plane effective-area estimate
1,Target estimate (1.28 Vpp),0.044413,0.060749,0.000016,0.000027,7.145147e-09,279.986398,279.986398,pressure_or_pii_minus_6dB,Gaussian focal-plane effective-area estimate


A_eff = 0.0444128 mm²; power is estimated from I_peak × A_eff under the stated Gaussian-profile assumption.


## Pressure estimates at the target voltage

This section restores pressure estimates from the corrected 3f₀ pulse. It separates direct waveform scaling from intensity-derived equivalents:

- Direct peak values scale the measured corrected waveform by `target_voltage_vpp/reference_voltage_vpp`.
- `p_rms = sqrt(I × rho × c)` derives RMS pressure under the plane-wave assumption.
- `sqrt(2 × I_SPPA × rho × c)` is a sinusoidal peak-equivalent, not necessarily the maximum sample of the pulsed waveform.
- I_SPTA includes inter-pulse silence, so its pressure is only a temporal RMS-equivalent and is not a pulse-peak estimate.

In [65]:
# Reimplemented pressure estimates: corrected 3f0 reference and target-voltage values.
# Uses the 3f0 frequency/aperture-corrected waveform already calculated above.
reference_peak_compressional_3f0_pa = max(float(np.max(corrected_3f0_baseline_centred_pa)), 0.0)
reference_peak_rarefactional_3f0_pa = max(float(-np.min(corrected_3f0_baseline_centred_pa)), 0.0)
target_peak_compressional_scaled_pa = reference_peak_compressional_3f0_pa * pressure_scale
target_peak_rarefactional_scaled_pa = reference_peak_rarefactional_3f0_pa * pressure_scale

target_isppa_w_m2 = float(target_3f0_estimate_df.iloc[0]["I_SPPA_W_cm2"]) * 1e4
target_ispta_w_m2 = float(target_3f0_estimate_df.iloc[0]["I_SPTA_W_cm2"]) * 1e4

def pressure_equivalents_from_intensity(intensity_w_m2):
    """Return plane-wave RMS and sinusoidal peak-equivalent pressure in Pa."""
    if not np.isfinite(intensity_w_m2) or intensity_w_m2 < 0:
        return np.nan, np.nan
    pressure_rms_pa = np.sqrt(intensity_w_m2 * acoustic_impedance_pa_s_m)
    return pressure_rms_pa, np.sqrt(2.0) * pressure_rms_pa

(target_pressure_rms_from_isppa_pa,
 target_pressure_peak_equivalent_from_isppa_pa) = pressure_equivalents_from_intensity(target_isppa_w_m2)
(target_pressure_rms_from_ispta_pa,
 target_pressure_peak_equivalent_from_ispta_pa) = pressure_equivalents_from_intensity(target_ispta_w_m2)

pressure_estimates_df = pd.DataFrame([
    {
        "quantity": "Reference maximum compressional pressure",
        "source": "Corrected 3f0 waveform", "voltage_vpp": reference_voltage_vpp,
        "pressure_kpa": reference_peak_compressional_3f0_pa / 1000,
        "interpretation": "Maximum positive waveform sample",
    },
    {
        "quantity": "Reference maximum rarefactional magnitude",
        "source": "Corrected 3f0 waveform", "voltage_vpp": reference_voltage_vpp,
        "pressure_kpa": reference_peak_rarefactional_3f0_pa / 1000,
        "interpretation": "Maximum negative waveform magnitude",
    },
    {
        "quantity": "Target maximum compressional pressure",
        "source": "Linear waveform scaling", "voltage_vpp": target_voltage_vpp,
        "pressure_kpa": target_peak_compressional_scaled_pa / 1000,
        "interpretation": "Reference peak × V_target/V_reference",
    },
    {
        "quantity": "Target maximum rarefactional magnitude",
        "source": "Linear waveform scaling", "voltage_vpp": target_voltage_vpp,
        "pressure_kpa": target_peak_rarefactional_scaled_pa / 1000,
        "interpretation": "Reference magnitude × V_target/V_reference",
    },
    {
        "quantity": "Target RMS pressure from I_SPPA",
        "source": "sqrt(I_SPPA × rho × c)", "voltage_vpp": target_voltage_vpp,
        "pressure_kpa": target_pressure_rms_from_isppa_pa / 1000,
        "interpretation": "Within-pulse RMS-equivalent pressure",
    },
    {
        "quantity": "Target sinusoidal peak-equivalent from I_SPPA",
        "source": "sqrt(2 × I_SPPA × rho × c)", "voltage_vpp": target_voltage_vpp,
        "pressure_kpa": target_pressure_peak_equivalent_from_isppa_pa / 1000,
        "interpretation": "Peak-equivalent; not the exact pulsed-waveform maximum",
    },
    {
        "quantity": "Target temporal RMS pressure from I_SPTA",
        "source": "sqrt(I_SPTA × rho × c)", "voltage_vpp": target_voltage_vpp,
        "pressure_kpa": target_pressure_rms_from_ispta_pa / 1000,
        "interpretation": "Includes inter-pulse silence; not a pulse peak",
    },
])
display(pressure_estimates_df.style.format({"voltage_vpp": "{:.4g}", "pressure_kpa": "{:.6g}"}))


,quantity,source,voltage_vpp,pressure_kpa,interpretation
0,Reference maximum compressional pressure,Corrected 3f0 waveform,10,373.686,Maximum positive waveform sample
1,Reference maximum rarefactional magnitude,Corrected 3f0 waveform,10,379.621,Maximum negative waveform magnitude
2,Target maximum compressional pressure,Linear waveform scaling,1.28,47.8318,Reference peak × V_target/V_reference
3,Target maximum rarefactional magnitude,Linear waveform scaling,1.28,48.5914,Reference magnitude × V_target/V_reference
4,Target RMS pressure from I_SPPA,sqrt(I_SPPA × rho × c),1.28,29.9749,Within-pulse RMS-equivalent pressure
5,Target sinusoidal peak-equivalent from I_SPPA,sqrt(2 × I_SPPA × rho × c),1.28,42.391,Peak-equivalent; not the exact pulsed-waveform maximum
6,Target temporal RMS pressure from I_SPTA,sqrt(I_SPTA × rho × c),1.28,0.487799,Includes inter-pulse silence; not a pulse peak


## ISAPA: Gaussian spatial-average pulse-average intensity

IEC defines ISAPA as pulse-average intensity averaged over a specified beam area. Here the beam area is the focal-plane −6 dB PII/intensity contour, i.e. points with `I_PA/I_SPPA ≥ 0.25`.

For a separable Gaussian profile, the integral within this contour contains 75% of total Gaussian power. Therefore:

$$
\begin{align}
I_\mathrm{SAPA}=
\dfrac{0.75\,P_\mathrm{PA}}{A_{b,6}}
=I_\mathrm{SPPA}
\dfrac{0.75A_\mathrm{eff}}{A_{b,6}}.
\end{align}
$$

The same geometry gives ISATA from I_SPTA. This is a Gaussian-model estimate based on the focal widths, rather than a direct 2D hydrophone integration. [IEC ISAPA definition](https://cdn.standards.iteh.ai/samples/18526/7d76da2b009d4f8c917b3f1735987dff/IEC-62127-1-2007-AMD1-2013.pdf), [spatial-average calculation method](https://www.acoustics.co.uk/wp-content/uploads/2024/04/PPE-User-Guide-V2-0424.pdf).

In [66]:
# Gaussian-model ISAPA/ISATA using the same focal widths as the power estimate.
# Convert entered widths to full −6 dB PII/intensity widths defining A_b,6.
if beamwidth_definition == "pressure_or_pii_minus_6dB":
    beamwidth_x_6db_um = beamwidth_x_um
    beamwidth_y_6db_um = beamwidth_y_um
elif beamwidth_definition == "intensity_fwhm":
    # A Gaussian intensity profile reaches 25% at sqrt(2) times its half-maximum radius.
    beamwidth_x_6db_um = beamwidth_x_um * np.sqrt(2)
    beamwidth_y_6db_um = beamwidth_y_um * np.sqrt(2)
else:
    raise ValueError("Unknown beamwidth_definition")

beam_area_6db_m2 = np.pi * (beamwidth_x_6db_um * 1e-6) * (beamwidth_y_6db_um * 1e-6) / 4
beam_area_6db_mm2 = beam_area_6db_m2 * 1e6
beam_area_threshold_fraction = 0.25
fraction_of_gaussian_power_inside_beam_area = 1 - beam_area_threshold_fraction

isapa_rows = []
for _, power_row in power_estimate_df.iterrows():
    p_pa_total_w = float(power_row["pulse_average_power_W"])
    p_ta_total_w = float(power_row["temporal_average_power_W"])
    isapa_w_m2 = fraction_of_gaussian_power_inside_beam_area * p_pa_total_w / beam_area_6db_m2
    isata_w_m2 = fraction_of_gaussian_power_inside_beam_area * p_ta_total_w / beam_area_6db_m2
    isapa_rows.append({
        "voltage_case": power_row["voltage_case"],
        "A_b_6_mm2": beam_area_6db_mm2,
        "Gaussian_power_fraction_inside_A_b_6": fraction_of_gaussian_power_inside_beam_area,
        "I_SPPA_W_cm2": power_row["I_SPPA_W_cm2"],
        "I_SAPA_W_cm2": isapa_w_m2 / 1e4,
        "I_SPTA_W_cm2": power_row["I_SPTA_W_cm2"],
        "I_SATA_W_cm2": isata_w_m2 / 1e4,
        "beamwidth_x_6dB_um": beamwidth_x_6db_um,
        "beamwidth_y_6dB_um": beamwidth_y_6db_um,
        "method": "Gaussian integral inside focal −6 dB PII/intensity contour",
    })
isapa_estimate_df = pd.DataFrame(isapa_rows)
display(isapa_estimate_df)

print(
    f"A_b,6 = {beam_area_6db_mm2:.6g} mm²; Gaussian power within A_b,6 = "
    f"{fraction_of_gaussian_power_inside_beam_area:.0%} of total power."
)


,voltage_case,A_b_6_mm2,Gaussian_power_fraction_inside_A_b_6,I_SPPA_W_cm2,I_SAPA_W_cm2,I_SPTA_W_cm2,I_SATA_W_cm2,beamwidth_x_6dB_um,beamwidth_y_6dB_um,method
0,Reference (10 Vpp),0.061569,0.75,3.707816,2.005968,0.000982,0.000531,279.986398,279.986398,Gaussian integral inside focal −6 dB PII/inten...
1,Target estimate (1.28 Vpp),0.061569,0.75,0.060749,0.032866,0.000016,0.000009,279.986398,279.986398,Gaussian integral inside focal −6 dB PII/inten...


A_b,6 = 0.0615692 mm²; Gaussian power within A_b,6 = 75% of total power.
